In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
import shap
import matplotlib.pyplot as plt
from scipy.stats import scoreatpercentile
from tqdm import tqdm
import time

# 1. 파일 이름 정의
AMC_FILE_NAME = 'amc_tidy.csv'

amc = pd.read_csv(AMC_FILE_NAME, encoding= 'utf-8', dtype='str')

# 1. 파일 이름 정의
SMC_FILE_NAME = 'D2024_0087_02(op_id 포함).csv'

smc = pd.read_csv(SMC_FILE_NAME, encoding= 'utf-8', dtype='str')
smc['row_no'] = smc['row_no'].astype('int')
smc = smc.sort_values('row_no').reset_index(drop=True)

def process_age(age_str):
    age_str = str(age_str).strip()
    if '이상' in age_str:
        return 80
    elif '-' in age_str:
        return 20
    elif '대' in age_str:
        return int(age_str.replace('대', ''))
    return 0


smc['age'] = smc['age'].apply(process_age).astype(int)
smc = smc.drop(['op_id'], axis =1)

In [ ]:
# smc 데이터 전처리
threshold = 50
value_counts = smc['procedure_cd_fin'].value_counts()
frequent_codes = value_counts[value_counts >= threshold].index
smc = smc[smc['procedure_cd_fin'].isin(frequent_codes)]

columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float

dtype_int = {col: target_int for col in columns_to_int}
smc = smc.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
smc = smc.astype(dtype_float)

# ⭐ 사용할 컬럼 목록 정의 (20개)
ALL_COLUMNS_TO_USE = [
    'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_FEATURE = 'use_quan'

smc = smc[ALL_COLUMNS_TO_USE]

features = [c for c in ALL_COLUMNS_TO_USE if c != TARGET_FEATURE]

n_features = len(features)
print("--- 데이터 로드 및 전처리 완료 ---\n")


In [ ]:
# 1. IQR 및 임계값 계산
# -------------------------------------------------------------
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'

# use_quan이 0보다 큰 데이터만 사용하여 IQR 계산 (수혈을 받은 환자 대상)
# 0을 포함하여 계산하면 1.5*IQR 기준이 매우 낮아질 수 있습니다.
# 여기서는 일반적인 IQR 기준 적용을 위해 0을 포함하여 계산합니다.
Q1 = smc[TARGET_COLUMN].quantile(0.25)
Q3 = smc[TARGET_COLUMN].quantile(0.75)
IQR = Q3 - Q1

# IQR 상한 임계값 (Upper Bound)
iqr_upper_bound = Q3 + 1.5 * IQR

print(f"use_quan Q1: {Q1:.2f}")
print(f"use_quan Q3: {Q3:.2f}")
print(f"use_quan IQR: {IQR:.2f}")
print(f"use_quan IQR 상한 임계값 (Q3 + 1.5*IQR): {iqr_upper_bound:.2f}")

# -------------------------------------------------------------
# 2. 이상치 필터링 조건 생성
# -------------------------------------------------------------

# 조건 A: IQR 상한을 초과하는 경우
condition_A = (smc[TARGET_COLUMN] > iqr_upper_bound)

# 조건 B: MSBOS 값의 2배를 초과하는 경우
condition_B = (smc[TARGET_COLUMN] > 2 * smc[MSBOS_COLUMN])

# 최종 이상치 조건: 조건 A와 조건 B를 동시에 만족
# np.logical_and를 사용하여 두 조건이 모두 참인 행을 찾습니다.
outlier_condition = np.logical_and(condition_A, condition_B)

# -------------------------------------------------------------
# 3. 이상치 제거
# -------------------------------------------------------------

# 이상치로 간주되는 행의 개수 확인
num_outliers = outlier_condition.sum()

# 이상치가 아닌 데이터만 남김 (필터링된 데이터프레임)
smc_filtered = smc[~outlier_condition].copy()

print(f"\n제거된 이상치 데이터 개수 (조건 A AND 조건 B 만족): {num_outliers} 개")
print(f"원래 데이터셋 크기: {len(smc)} 행")
print(f"이상치 제거 후 데이터셋 크기: {len(smc_filtered)} 행")
smc_filtered['institution'] = 'smc'
# 원본 df를 사용하지 않고 df_filtered를 사용하여 이후 모델링을 진행합니다.

In [ ]:
# amc 데이터 전처리
amc.columns
amc = amc.drop(['pt_id', 'op_id', 'anesthesia_cd', 'anesthesia_nm2'], axis=1).rename(columns = {'anesthesia_cd2':'anesthesia_cd',
                                                                                                             'use_quan_imputed':'use_quan'})

columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float

dtype_int = {col: target_int for col in columns_to_int}
amc = amc.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
amc = amc.astype(dtype_float)

# ⭐ 사용할 컬럼 목록 정의 (20개)
ALL_COLUMNS_TO_USE = [
    'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_FEATURE = 'use_quan'

amc = amc[ALL_COLUMNS_TO_USE]

features = [c for c in ALL_COLUMNS_TO_USE if c != TARGET_FEATURE]

n_features = len(features)
print("--- 데이터 로드 및 전처리 완료 ---\n")
amc.info()




In [ ]:
# 1. IQR 및 임계값 계산
# -------------------------------------------------------------
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'

# use_quan이 0보다 큰 데이터만 사용하여 IQR 계산 (수혈을 받은 환자 대상)
# 0을 포함하여 계산하면 1.5*IQR 기준이 매우 낮아질 수 있습니다.
# 여기서는 일반적인 IQR 기준 적용을 위해 0을 포함하여 계산합니다.
Q1 = amc[TARGET_COLUMN].quantile(0.25)
Q3 = amc[TARGET_COLUMN].quantile(0.75)
IQR = Q3 - Q1

# IQR 상한 임계값 (Upper Bound)
iqr_upper_bound = Q3 + 1.5 * IQR

print(f"use_quan Q1: {Q1:.2f}")
print(f"use_quan Q3: {Q3:.2f}")
print(f"use_quan IQR: {IQR:.2f}")
print(f"use_quan IQR 상한 임계값 (Q3 + 1.5*IQR): {iqr_upper_bound:.2f}")

# -------------------------------------------------------------
# 2. 이상치 필터링 조건 생성
# -------------------------------------------------------------

# 조건 A: IQR 상한을 초과하는 경우
condition_A = (amc[TARGET_COLUMN] > iqr_upper_bound)

# 조건 B: MSBOS 값의 2배를 초과하는 경우
condition_B = (amc[TARGET_COLUMN] > 2 * amc[MSBOS_COLUMN])

# 최종 이상치 조건: 조건 A와 조건 B를 동시에 만족
# np.logical_and를 사용하여 두 조건이 모두 참인 행을 찾습니다.
outlier_condition = np.logical_and(condition_A, condition_B)

# -------------------------------------------------------------
# 3. 이상치 제거
# -------------------------------------------------------------

# 이상치로 간주되는 행의 개수 확인
num_outliers = outlier_condition.sum()

# 이상치가 아닌 데이터만 남김 (필터링된 데이터프레임)
amc_filtered = amc[~outlier_condition].copy()

print(f"\n제거된 이상치 데이터 개수 (조건 A AND 조건 B 만족): {num_outliers} 개")
print(f"원래 데이터셋 크기: {len(amc)} 행")
print(f"이상치 제거 후 데이터셋 크기: {len(amc_filtered)} 행")
amc_filtered['institution'] = 'amc'
# 원본 df를 사용하지 않고 df_filtered를 사용하여 이후 모델링을 진행합니다.

In [ ]:
pd.concat([smc_filtered, amc_filtered]).reset_index(drop= 1).to_csv('integration_data.csv')